In [1]:
# 1) GPU 켜기: 런타임 → 런타임 유형 변경 → GPU
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 112.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18


In [2]:
from google.colab import userdata
from roboflow import Roboflow

api_key = userdata.get('ROBOFLOW_API_KEY')  # 각자 본인 키
rf = Roboflow(api_key=api_key)
project = rf.workspace("s-workspace-pv7pr").project("findeye_osp")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to FINDEYE_OSP-1 in yolov11:: 100%|██████████| 2043/2043 [00:00<00:00, 3421.80it/s]


 # 모델 크기 비교 학습

In [3]:
from ultralytics import YOLO

# 전원 공통 고정값
DATA     = f"{dataset.location}/data.yaml"
EPOCHS   = 100
IMGSZ    = 640
BATCH    = 16
PATIENCE = 20

# B가 바꾸는 유일한 변수 = 모델 크기
for size in ["yolo11n.pt", "yolo11s.pt"]:
    model = YOLO(size)
    model.train(
        data=DATA,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        hsv_v=0.1,                              # 명도가 분류 기준 → 낮게 고정
        name=f"size_{size.replace('.pt','')}"   # 결과 폴더 분리
    )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/FINDEYE_OSP-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.

# 결과 추출

In [4]:
import pandas as pd

for size in ["yolo11n", "yolo11s"]:
    df = pd.read_csv(f"runs/detect/size_{size}/results.csv")
    df.columns = df.columns.str.strip()
    last = df.iloc[-1]                        # 마지막 epoch 행
    print(
        size,
        "mAP50:",    round(last["metrics/mAP50(B)"], 4),
        "mAP50-95:", round(last["metrics/mAP50-95(B)"], 4),
        "Recall:",   round(last["metrics/recall(B)"], 4),
        "time(s):",  round(last["time"], 1),
    )

yolo11n mAP50: 0.9859 mAP50-95: 0.8008 Recall: 0.9626 time(s): 1453.6
yolo11s mAP50: 0.9682 mAP50-95: 0.7789 Recall: 0.9857 time(s): 1775.7
